<a target="_blank" href="https://colab.research.google.com/github/ZHAW-ZAV/TSO-FS26-students/blob/main/03_forecasting/03_01_forecasting.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
import pandas as pd
import numpy as np

from plotly import graph_objects as go
from plotly import express as px
from plotly.subplots import make_subplots

from statsmodels.tsa.holtwinters import ExponentialSmoothing

import sys
import os

IN_COLAB = "google.colab" in sys.modules

file_id_atm = "1pZnA_C3LKTvpYA4_Dzqw1bXQDryy7q9c"

if IN_COLAB:
    path_to_atm = "/content/data/dailyATM.csv"
    os.makedirs(os.path.dirname(path_to_atm), exist_ok=True)
    !gdown "https://drive.google.com/uc?id={file_id_atm}" -O "{path_to_atm}"
else:
    import gdown

    url = f"https://drive.google.com/uc?id={file_id_atm}"
    path_to_atm = "data/dailyATM.csv"
    os.makedirs(os.path.dirname(path_to_atm), exist_ok=True)
    gdown.download(url, path_to_atm, quiet=False)


The code above loads the data, don't modify.

---------------

***Notebook starts here***

# Import data from CSV file and plot

In [ ]:
# Import data. Note: we set column 'timestamp' as the index of this df!
df = pd.read_csv(
    path_to_atm,
    parse_dates=["timestamp"],
    index_col="timestamp"
)

# make sure index is datetime
df.index = pd.to_datetime(df.index)

# Enforce daily frequency (this fills values for arrivals and departure on missing days with NaN)
df = df.asfreq("D")

# Display first 5 rows of df
df.head()

In [ ]:
# Plot the data (number of arrivals and departures at an airport per day)

fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["arrivals"],
        mode="lines",
        name="Arrivals",
    )
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["departures"],
        mode="lines",
        name="Departures",
    )
)

fig.update_layout(
    height=600,
    width=1000,
    xaxis_title="Time",
    yaxis_title="Daily Movements",
)

fig.show()

#fig.write_image("daily_movements.pdf")

# Resample data to weekly

In [ ]:
df_weekly = df.resample("W").sum()
df_weekly.head(10)

In [ ]:
df_weekly_avg = df.resample("W").mean()
df_weekly_avg.head(10)

# Decomposition

In [ ]:
# Decomposition of daily number of arrivals / departures into trend, seasonal component, and residual
from statsmodels.tsa.seasonal import seasonal_decompose
decompose = seasonal_decompose(df["departures"], model="additive", period=24)


# Create a 4-row subplot figure
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                    subplot_titles=["Observed", "Trend", "Seasonal", "Residual"])

# Add traces for each component
fig.add_trace(go.Scatter(x=decompose.observed.index, y=decompose.observed, 
                         mode='lines', line=dict(color='black', width=2), name="Observed"), row=1, col=1)

fig.add_trace(go.Scatter(x=decompose.trend.index, y=decompose.trend, 
                         mode='lines', line=dict(color='blue', width=2), name="Trend"), row=2, col=1)

fig.add_trace(go.Scatter(x=decompose.seasonal.index, y=decompose.seasonal, 
                         mode='lines', line=dict(color='green', width=2), name="Seasonal"), row=3, col=1)

fig.add_trace(go.Scatter(x=decompose.resid.index, y=decompose.resid, 
                         mode='lines', line=dict(color='cyan', width=2), name="Residual"), row=4, col=1)

# Update layout
fig.update_layout(
    height=800, width=1000,
    showlegend=False,
    title_text="Time Series Decomposition",
    xaxis4_title="Date"
)

# Enable grid lines
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

fig.show()

# Basic Forecasting Methods

Let us apply some basic forecasting methods to the daily number of arrivals time series. More specifically, we generate a **30-day forecast** of daily arrivals using three simple benchmark forecasting methods:

- **Naive method**
- **Mean (average) method**
- **Drift method**

In [ ]:
# Forecast Horizon of 30 days
forecast_horizon = 30

# Create a forecast index
forecast_index = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), 
                               periods=forecast_horizon, 
                               freq='D')

# Naive Method
forecast_naive = [df['arrivals'].iloc[-1]] * forecast_horizon
forecast_naive = pd.Series(forecast_naive, index=forecast_index)

# Average Method
forecast_mean = [df['arrivals'].mean()] * forecast_horizon
forecast_mean = pd.Series(forecast_mean, index=forecast_index)

# Drift Method
y_T = df['arrivals'].iloc[-1]   # Last observed value
y_1 = df['arrivals'].iloc[0]    # First observed value
T = len(df['arrivals'])    # Number of observations
drift = y_T + (y_T - y_1) / (T - 1) * np.arange(1, forecast_horizon + 1)
drift_forecast = pd.Series(drift, index=forecast_index)


# Plot the combined data
fig = go.Figure()

# Add historical data
fig.add_trace(go.Scatter(x=df.index, y=df["arrivals"],
                         mode="lines+markers", name="Observations", line=dict(color='blue')))

# Add forecast data
fig.add_trace(go.Scatter(x=forecast_naive.index, y=forecast_naive,
                         mode="lines+markers", name="Naive Forecast", line=dict(color='red', dash="dash")))

fig.add_trace(go.Scatter(x=forecast_mean.index, y=forecast_mean,
                         mode="lines+markers", name="Mean Forecast", line=dict(color='green', dash="dash")))

fig.add_trace(go.Scatter(x=drift_forecast.index, y=drift_forecast,
                         mode="lines+markers", name="Drift Forecast", line=dict(color='violet', dash="dash")))

# Update layout
fig.update_layout(
    title="Number of Daily Arrivals with Naive Forecast",
    xaxis_title="Time",
    yaxis_title="Number of Arrivals per Day",
    font=dict(size=16),
    showlegend=True
)

fig.show()

# Simple Exponential Smoothing

### Data Preparation
Simple Exponential Smoothing should only be applied to time series which fluctuate around a constant level (i.e, time series which do not experience a trend or seasonality). This is not the case for number of daily arrivals vs. time (see plot above).

In order to apply simple exponential smoothing, we first calculate the change in arrivals over time (which we add as a new column `df[arrivals_change]`).

In [ ]:
df['arrivals_change']=df['arrivals'].diff()

# Create plot of df['arrivals_change']
fig = px.line(df, x=df.index, y="arrivals_change", markers=True, 
              title="Daily Change in Number of Arrivals")

# Update layout
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Daily Change in Number of Arrivals",
    font=dict(size=16),
    showlegend=True
)

fig.show()

### Application of Simple Exponential Smoothing

In the following example, we set parameter $\alpha = 0.3$ simply as an **illustrative value** to show how Simple Exponential Smoothing works. The magnitude of $\alpha$ affects the model output as follows:
- A small value of $\alpha$ (e.g. 0.05 to 0.2) → strong smoothing (slow adaptation, meaning forecast is more stable but may *lag* behind sudden changes)
- A large value of $\alpha$ (e.g. 0.7 to 0.95) → very reactive to recent observations, forecast can become *more sensitive/noisy* (i.e., risk of overreacting to random fluctuations)



In practice, the value of $\alpha$ is usually *estimated from the data* by minimizing a forecast error criterion (most commonly the sum of squared errors) over the in-sample fitted values. With `statsmodels`, you can let the model "find" the optimal value of $\alpha$ in an automatical way by setting `optimized=True` (and not fixing `smoothing_level`) as follows:

```python
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

model = SimpleExpSmoothing(
    df['arrivals_change'].dropna(),
    initialization_method="heuristic"
)

fitted_model = model.fit(optimized=True)
```

In [ ]:

# Define a value for parameter alpha (must be within 0 and 1)
alpha = 0.3

# Setup Model & Fit Model
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

model = SimpleExpSmoothing(
    df['arrivals_change'].dropna(), 
    initialization_method="heuristic"
    )

fitted_model = model.fit(smoothing_level=alpha, optimized=False)


# Forecast the next 30 days
forecast_horizon = 30
forecast = fitted_model.forecast(forecast_horizon)

# Create a forecast index
forecast_index = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), 
                               periods=forecast_horizon, 
                               freq='D')

# Convert forecast to a Pandas Series
forecast_series = pd.Series(forecast, index=forecast_index)

# Plot the Data (Observations and Forecast)
fig = go.Figure()

# Observations
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["arrivals_change"],
        mode="lines+markers",
        name="Observations",
        line=dict(color="blue", width=2),
        marker=dict(size=6),
    )
)

# Fitted values
fig.add_trace(
    go.Scatter(
        x=fitted_model.fittedvalues.index,
        y=fitted_model.fittedvalues,
        mode="lines",
        name="Fitted Values",
        line=dict(color="black", width=2, dash="dash"),
    )
)

# Forecast
fig.add_trace(
    go.Scatter(
        x=forecast_series.index,
        y=forecast_series,
        mode="lines",
        name="Forecast",
        line=dict(color="red", width=2),
    )
)

fig.update_layout(
    height=600,
    width=1000,
    xaxis_title="Time",
    yaxis_title="Daily Change Arrivals",
    title=rf"Exponential Smoothing, α = {alpha}",
    font=dict(size=16),
    legend=dict(font_size=14),
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

fig.show()

# Exponential Smoothing with Trend

### Data Preparation
To apply exponential smoothing with trend, we need a time series, which has a (clear) trend. This is not the case for the entire number of arrivals vs. time time series. For this case, let us filter the original time series for a time period, in which we see such a trend first.

In [ ]:
# Filter Time Series (we want a time series that exhibits a "clear" trend)
df_filtered = df.loc["2024-01-08":"2024-03-23"]

# Plot the combined data
fig = go.Figure()

# Add historical data
fig.add_trace(go.Scatter(x=df_filtered.index, y=df_filtered['arrivals'],
                         mode="lines+markers", name="Observations", line=dict(color='blue')))

fig.update_layout(
    height=600,
    width=1000,
    xaxis_title="Time",
    yaxis_title="Daily Number of Arrivals",
    font=dict(size=16),
    legend=dict(font_size=14),
)

### Application of Exponential Smoothing with Trend (Holt's Method)

In the following example, we specify the smoothing parameters $\alpha = 0.7$ and $\beta = 0.005$ as **illustrative values** to demonstrate how Exponential Smoothing with Trend (Holt's method) works.

Here:

- $ \alpha $ controls the smoothing of the **level** (small $\alpha$ → smoother level, slower reaction; large $\alpha$ → reacts strongly to recent observations)
- $ \beta $ controls the smoothing of the **trend** (small $\beta$ → stable trend estimate; large $\beta$ → trend reacts quickly to new changes)

In practice, these parameters are usually **estimated from the data** by minimizing an in-sample forecast error measure (most commonly the sum of squared errors).

With `statsmodels`, you can let the model automatically determine the optimal values of $ \alpha $ and $ \beta $ by setting `optimized=True` and not manually specifying the smoothing parameters.

Example:

```python
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    df_filtered['arrivals'],
    trend="add",
    seasonal=None,
    initialization_method="estimated"
)

fitted_model = model.fit(optimized=True)


In [ ]:
# Define Model Parameters
alpha = 0.7
beta = 0.005

# Fit the model with additive trend and multiplicative seasonality
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    df_filtered['arrivals'],
    trend="add",
    seasonal=None,
    initialization_method="known",                  # Use known initialization, since we (pretend to) know the parameter values
    initial_level=df_filtered['arrivals'].iloc[0],  # Provide initial level directly
    initial_trend=0                                 # Provide initial trend directly
)


# Fit the model with specified smoothing parameters
fitted_model = model.fit(
      smoothing_level=alpha,    # Alpha: Level smoothing
      smoothing_trend=beta,     # Beta: Trend smoothing (Replaces smoothing_slope)
      optimized=False           # Set to False since we specify parameters
)

# Forecast for the next 34 days
forecast_horizon = 34
forecast = fitted_model.forecast(forecast_horizon)

# Create a forecast index
forecast_index = pd.date_range(start=df_filtered['arrivals'].index[-1] + pd.Timedelta(days=1),
                               periods=forecast_horizon, 
                               freq='D')

# Convert the forecast to a Pandas Series
forecast_series = pd.Series(forecast, index=forecast_index)

# Plot the combined data
fig = go.Figure()

# Add historical data
fig.add_trace(go.Scatter(x=df_filtered.index, y=df_filtered['arrivals'],
                         mode="lines+markers", name="Observations", line=dict(color='blue')))

# Add forecast data
fig.add_trace(go.Scatter(x=forecast_series.index, y=forecast_series,
                         mode="lines+markers", name="Forecast", line=dict(color='red')))

# Add the ground truth for the forecast period
df_ground_truth = df.loc["2024-03-24":"2024-04-26"]
fig.add_trace(go.Scatter(x=df_ground_truth.index, y=df_ground_truth['arrivals'],
                         mode="lines+markers", name="Ground Truth", line=dict(color='black', dash="dash")))

# Update layout
fig.update_layout(
    title_text=f"Exponential Smoothing on Daily Arrivals, α={alpha:.2f}, β={beta:.2e}",
    xaxis_title="Time",
    yaxis_title="Daily Arrivals",
    font=dict(size=16),
    showlegend=True
)

fig.show()

# Exponential Smoothing with Trend and Seasonality (Holt-Winters Method)

In the following example, we specify the smoothing parameters $\alpha = 0.31$, $\beta = 1.37 \times 10^{-9}$, and $\gamma = 0.60$ as **illustrative values** to demonstrate how Holt–Winters Exponential Smoothing with additive trend and additive seasonality works. In practice, $\alpha$, $\beta$, and $\gamma$ are usually **estimated automatically** by minimizing an in-sample error criterion (typically the sum of squared errors). With `statsmodels`, you can let the model determine the optimal parameters by setting:

```python
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    df_filtered['arrivals'],
    trend="add",
    seasonal="add",
    seasonal_periods=7,
    initialization_method="estimated"
)

fitted_model = model.fit(optimized=True)

In [ ]:
# Define parameter values
alpha = 0.31
beta = 1.37e-09
gamma = 0.60

# Apply Holt-Winters' Exponential Smoothing
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    df_filtered['arrivals'],
    trend="add",
    seasonal="add",          
    seasonal_periods=7,      # Weekly seasonality
    initialization_method="estimated"
)

# Fit the model
fitted_model = model.fit(
    smoothing_level=alpha,      # Alpha: Level smoothing
    smoothing_trend=beta,       # Beta: Trend smoothing
    smoothing_seasonal=gamma,   # Gamma: Seasonal smoothing
    optimized=False             # Set to False since we specify the parameters
)

# Forecast for the next 34 days
forecast_horizon = 34
forecast = fitted_model.forecast(forecast_horizon)

# Create a forecast index
forecast_index = pd.date_range(
    start=df_filtered.index[-1] + pd.Timedelta(days=1),
    periods=forecast_horizon,
    freq="D"
)

# Convert the forecast to a Pandas Series
forecast_series = pd.Series(forecast, index=forecast_index)

# Plot the combined data
fig = go.Figure()

# Add historical data
fig.add_trace(go.Scatter(x=df_filtered.index, y=df_filtered['arrivals'],
                         mode="lines+markers", name="Observations", line=dict(color='blue')))

# Add forecast data
fig.add_trace(
    go.Scatter(
        x=forecast_series.index,
        y=forecast_series,
        mode="lines+markers",
        name="Forecast",
        line=dict(color="red")
    )
)

# Add the ground truth for the forecast period
df_ground_truth = df.loc["2024-03-24":"2024-04-26"]
fig.add_trace(go.Scatter(x=df_ground_truth.index, y=df_ground_truth['arrivals'],
                         mode="lines+markers", name="Ground Truth", line=dict(color='black', dash="dash")))

# Update layout
fig.update_layout(
    title=f'Exponential Smoothing with Trend and Seasonality, α={alpha}, β={beta}',
    xaxis_title="Year",
    yaxis_title="Daily Arrivals",
    font=dict(size=16),
    showlegend=True
)

fig.show()